In [47]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

for all notebooks
import os
from pathlib import Path
# Always run from project root
if Path(os.getcwd()).name == 'notebooks':
    os.chdir(Path('../').resolve())

In [48]:
# DFDC already has real/fake — just point config to train folder
# Let's count what we have
dfdc_real = list(Path('../data/dfdc/train/real').glob('*.png'))
dfdc_fake = list(Path('../data/dfdc/train/fake').glob('*.png'))

print(f'DFDC real images : {len(dfdc_real)}')
print(f'DFDC fake images : {len(dfdc_fake)}')

DFDC real images : 0
DFDC fake images : 0


In [49]:
def extract_frames(video_path, output_dir, max_frames=20):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // max_frames)
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    saved, idx = 0, 0
    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % step == 0:
            cv2.imwrite(f"{output_dir}/frame_{saved:04d}.jpg", frame)
            saved += 1
        idx += 1
    cap.release()
    return saved

# FF++ paths
fake_videos = list(Path('../data/faceforensics/manipulated_sequences').rglob('*.mp4'))
real_videos = list(Path('../data/faceforensics/original_sequences').rglob('*.mp4'))

print(f'FF++ fake videos : {len(fake_videos)}')
print(f'FF++ real videos : {len(real_videos)}')

# Extract frames — first 200 videos each (enough for training)
print('\nExtracting fake frames...')
for v in tqdm(fake_videos[:200]):
    extract_frames(v, f'../data/faceforensics/fake/{v.stem}')

print('\nExtracting real frames...')
for v in tqdm(real_videos[:200]):
    extract_frames(v, f'../data/faceforensics/real/{v.stem}')

print('\nDone!')

FF++ fake videos : 0
FF++ real videos : 0

Extracting fake frames...


0it [00:00, ?it/s]



Extracting real frames...


0it [00:00, ?it/s]


Done!


In [50]:
ff_real = list(Path('../data/faceforensics/real').rglob('*.jpg'))
ff_fake = list(Path('../data/faceforensics/fake').rglob('*.jpg'))
dfdc_real = list(Path('../data/dfdc/train/real').glob('*.png'))
dfdc_fake = list(Path('../data/dfdc/train/fake').glob('*.png'))

print('Final dataset summary:')
print(f'  FF++  real frames : {len(ff_real)}')
print(f'  FF++  fake frames : {len(ff_fake)}')
print(f'  DFDC  real images : {len(dfdc_real)}')
print(f'  DFDC  fake images : {len(dfdc_fake)}')
print(f'  Total             : {len(ff_real)+len(ff_fake)+len(dfdc_real)+len(dfdc_fake)}')

Final dataset summary:
  FF++  real frames : 0
  FF++  fake frames : 0
  DFDC  real images : 0
  DFDC  fake images : 0
  Total             : 0


In [51]:
import os
from pathlib import Path

# Check exact structure of what we have
print("DFDC structure:")
for p in sorted(Path('../data/dfdc').rglob('*'))[:10]:
    print(f"  {p}")

print("\nFF++ structure:")
for p in sorted(Path('../data/faceforensics').rglob('*'))[:10]:
    print(f"  {p}")

DFDC structure:

FF++ structure:


In [52]:
import yaml

cfg = {
    'data': {
        'dataset': 'both',
        'dfdc_root': 'data/dfdc/train',
        'ff_root': 'data/faceforensics',
        'num_workers': 0,
        'split_ratios': [0.70, 0.15, 0.15]
    },
    'preprocess': {
        'face_size': 224,
        'fft_size': 224,
        'face_margin': 0.3,
        'use_mtcnn': False
    },
    'model': {
        'dropout': 0.3,
        'freq_channels': [32, 64, 128],
        'fusion': 'attention_gate',
        'spatial_backbone': 'efficientnet_b0',
        'spatial_pretrained': True
    },
    'train': {
        'amp': True,
        'batch_size': 32,
        'early_stopping': 7,
        'epochs': 30,
        'lr': 0.0001,
        'lr_scheduler': 'cosine',
        'seed': 42,
        'warmup_epochs': 2,
        'weight_decay': 0.0001
    },
    'eval': {
        'metrics': ['auc', 'accuracy', 'f1'],
        'threshold': 0.5
    },
    'logging': {
        'checkpoint_dir': 'checkpoints',
        'project_name': 'deepfake-detection',
        'results_dir': 'results',
        'save_best_only': True,
        'use_wandb': False
    }
}

with open('../config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Verify by reading it back
with open('../config.yaml') as f:
    check = yaml.safe_load(f)

print('config.yaml written and verified!')
print(f"  dataset   : {check['data']['dataset']}")
print(f"  dfdc_root : {check['data']['dfdc_root']}")
print(f"  ff_root   : {check['data']['ff_root']}")
print(f"  batch_size: {check['train']['batch_size']}")

config.yaml written and verified!
  dataset   : both
  dfdc_root : data/dfdc/train
  ff_root   : data/faceforensics
  batch_size: 32


In [53]:
import yaml, sys, os
from pathlib import Path

sys.path.insert(0, '../')

# Get absolute project root
project_root = Path('../').resolve()
print(f'Project root: {project_root}')

# Verify folders exist
dfdc_real = project_root / 'data/dfdc/train/real'
dfdc_fake = project_root / 'data/dfdc/train/fake'
ff_real   = project_root / 'data/faceforensics/real'
ff_fake   = project_root / 'data/faceforensics/fake'

print(f'DFDC real exists : {dfdc_real.exists()} — {dfdc_real}')
print(f'DFDC fake exists : {dfdc_fake.exists()} — {dfdc_fake}')
print(f'FF++ real exists : {ff_real.exists()} — {ff_real}')
print(f'FF++ fake exists : {ff_fake.exists()} — {ff_fake}')

Project root: D:\
DFDC real exists : False — D:\data\dfdc\train\real
DFDC fake exists : False — D:\data\dfdc\train\fake
FF++ real exists : False — D:\data\faceforensics\real
FF++ fake exists : False — D:\data\faceforensics\fake


In [54]:
import yaml, sys, os
from pathlib import Path

# Use absolute path directly — no relative paths
PROJECT_ROOT = Path('D:/cpe646-deepfake')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Working directory: {os.getcwd()}')

# Verify folders
print(f'DFDC real : {(PROJECT_ROOT / "data/dfdc/train/real").exists()}')
print(f'DFDC fake : {(PROJECT_ROOT / "data/dfdc/train/fake").exists()}')
print(f'FF++ real : {(PROJECT_ROOT / "data/faceforensics/real").exists()}')
print(f'FF++ fake : {(PROJECT_ROOT / "data/faceforensics/fake").exists()}')

# Update config
with open(PROJECT_ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dfdc_root'] = 'data/dfdc/train'
cfg['data']['ff_root']   = 'data/faceforensics'
cfg['data']['dataset']   = 'both'

with open(PROJECT_ROOT / 'config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Load real data
from dataset import get_dataloaders
train_loader, val_loader, test_loader = get_dataloaders(cfg, use_dummy=False)

batch = next(iter(train_loader))
print('\nReal batch loaded!')
print(f'  spatial : {batch["spatial"].shape}')
print(f'  freq    : {batch["freq"].shape}')
print(f'  labels  : {batch["label"]}')
print(f'  sample  : {batch["path"][0]}')

Working directory: D:\cpe646-deepfake
DFDC real : True
DFDC fake : True
FF++ real : True
FF++ fake : True
  [DataLoader] Train: 65697 | Val: 14077 | Test: 14079

Real batch loaded!
  spatial : torch.Size([32, 3, 224, 224])
  freq    : torch.Size([32, 1, 224, 224])
  labels  : tensor([0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 1, 1, 1])
  sample  : data\dfdc\train\real\bzythlfnhq_14_0.png
